Один раз подключаем GDrive, если ещё не сделано. Там уже должен находиться датасет (wav, 22050Hz, 16бит, моно) и прочие используемые данные.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get update
!apt-get install -y build-essential cmake ninja-build

In [ ]:
!git clone https://github.com/OHF-voice/piper1-gpl.git
%cd piper1-gpl

In [ ]:
!python3 -m venv .venv --without-pip
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!.venv/bin/python3 get-pip.py

In [ ]:
!.venv/bin/pip install --upgrade pip
!.venv/bin/python -m pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!.venv/bin/pip install -e .[train]

In [ ]:
!bash build_monotonic_align.sh

In [ ]:
!.venv/bin/pip install scikit-build

In [ ]:
!.venv/bin/python setup.py build_ext --inplace

In [ ]:
!ls /content/piper1-gpl/src/piper/train/vits/monotonic_align

[Опциональное действие] Исправление фонем, предварительно скачать с репозитория https://github.com/mitrokun/espeak-ng-data папку (с 5 файлами) espeak-ng-data на gdrive, после чего запустить копирование данных. Если не планируете вмешиваться в piper в дальнейшем, то пропустите шаг.

In [10]:
!cp -r /content/drive/MyDrive/espeak-ng-data/* /content/piper1-gpl/src/piper/espeak-ng-data/

После первых эксперементов c ручной вознёй, показанной на ютубе и в piper1.ipynb можно перейти на автоматическое сохраниение последнего чекпоинта сразу на drive. А тажже на автосоздание onnx. Здесь пример продолжающий обучение:

Эту ячейку запускаем без изменений

In [ ]:
%%writefile /content/onnx_callback.py
import logging
import copy
import shutil
import json
from pathlib import Path
from typing import Optional

import torch
from lightning.pytorch.callbacks import Callback

_LOGGER = logging.getLogger("lightning.pytorch")

class OnnxExportCallback(Callback):
    """Кастомный колбэк для автоматического экспорта в ONNX 
    с поддержкой смещения эпох (epoch_offset).
    """
    def __init__(self, output_dir: str = "onnx_exports", epoch_offset: int = 0):
        super().__init__()
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.epoch_offset = epoch_offset

    def on_validation_epoch_end(self, trainer, pl_module):
        # 1. Пропускаем предварительную проверку (sanity check)
        if trainer.sanity_checking:
            return

        # 2. Экспортируем только на главном процессе (актуально для Multi-GPU/DDP)
        if trainer.global_rank != 0:
            return

        # Складываем текущую эпоху и заданное смещение
        epoch = trainer.current_epoch + self.epoch_offset
        
        # 3. Динамически забираем данные из запущенного Datamodule
        datamodule = getattr(trainer, "datamodule", None)
        voice_name = getattr(datamodule, "voice_name", "ru_RU-fr-medium")
        config_path = getattr(datamodule, "config_path", None)

        # 4. Форматируем имя файла под шаблон ru_RU-fr6400-medium.onnx
        parts = voice_name.split("-")
        if len(parts) == 3:
            onnx_filename = f"{parts[0]}-{parts[1]}{epoch}-{parts[2]}.onnx"
        else:
            onnx_filename = f"{voice_name}_{epoch}.onnx"

        onnx_path = self.output_dir / onnx_filename
        _LOGGER.info(f"Starting automatic ONNX export to Google Drive: {onnx_path}...")

        try:
            # 5. Клонируем генератор во избежание порчи весов обучающейся модели
            model_g = copy.deepcopy(pl_module.model_g)
            model_g.eval()
            
            # Переносим копию на CPU для безопасности и экономии GPU-памяти
            model_g = model_g.to("cpu")

            with torch.no_grad():
                model_g.dec.remove_weight_norm()

            # Точно копируем сигнатуру infer_forward из export_onnx.py
            def infer_forward(text, text_lengths, scales, sid=None):
                noise_scale = scales[0]
                length_scale = scales[1]
                noise_scale_w = scales[2]
                audio = model_g.infer(
                    text,
                    text_lengths,
                    noise_scale=noise_scale,
                    length_scale=length_scale,
                    noise_scale_w=noise_scale_w,
                    sid=sid,
                )[0].unsqueeze(1)

                return audio * 0.7

            model_g.forward = infer_forward

            num_symbols = model_g.n_vocab
            num_speakers = model_g.n_speakers

            # Подготовка dummy-данных
            dummy_input_length = 50
            sequences = torch.randint(
                low=0, high=num_symbols, size=(1, dummy_input_length), dtype=torch.long
            )
            sequence_lengths = torch.LongTensor([sequences.size(1)])

            sid: Optional[torch.LongTensor] = None
            if num_speakers > 1:
                sid = torch.LongTensor([0])

            scales = torch.FloatTensor([0.667, 1.0, 0.8])
            dummy_input = (sequences, sequence_lengths, scales, sid)

            # 6. Экспорт в ONNX
            torch.onnx.export(
                model=model_g,
                args=dummy_input,
                f=str(onnx_path),
                verbose=False,
                opset_version=15,
                input_names=["input", "input_lengths", "scales", "sid"],
                output_names=["output"],
                dynamic_axes={
                    "input": {0: "batch_size", 1: "phonemes"},
                    "input_lengths": {0: "batch_size"},
                    "output": {0: "batch_size", 2: "time"},
                },
            )
            _LOGGER.info(f"ONNX model successfully saved: {onnx_path}")

            # 7. Чтение оригинального конфига, исправление языка и сохранение на GDrive
            if config_path:
                config_src = Path(config_path)
                if config_src.exists():
                    config_dst = self.output_dir / f"{onnx_filename}.json"
                    
                    try:
                        with open(config_src, "r", encoding="utf-8") as f:
                            config_data = json.load(f)
                        
                        # Если ключа "language" нет, автоматически внедряем его
                        if "language" not in config_data:
                            lang_code = "ru-RU"
                            if len(parts) > 0:
                                lang_code = parts[0].replace("_", "-")
                            
                            config_data["language"] = {
                                "code": lang_code
                            }
                            _LOGGER.info(f"Auto-injected missing language config: 'language': {{'code': '{lang_code}'}}")
                        
                        # Записываем обновленный JSON файл
                        with open(config_dst, "w", encoding="utf-8") as f:
                            json.dump(config_data, f, indent=2, ensure_ascii=False)
                        
                        _LOGGER.info(f"Config JSON successfully processed and saved to: {config_dst}")
                    except Exception as json_err:
                        _LOGGER.error(f"Failed to process and inject language to JSON: {json_err}")
                        shutil.copy(config_src, config_dst)
                else:
                    _LOGGER.warning(f"Original config not found at {config_path}, JSON copy skipped.")

        except Exception as e:
            _LOGGER.error(f"Failed to auto-export ONNX at epoch {epoch}: {e}", exc_info=True)

Здесь указываете путь и имя для сохранения крайнего чекпоинта и путь для экспорта onnx моделей.  Перед стартом обучения в dirpath не должно быть filename, иначе логика сломается - переменуйте существующий чекпоинт.

In [ ]:
%%writefile /content/callbacks.yaml
trainer:
  callbacks:
    - class_path: lightning.pytorch.callbacks.ModelCheckpoint
      init_args:
        dirpath: "/content/drive/MyDrive/pt/fr"
        filename: "latest_fr"
        auto_insert_metric_name: false
        save_top_k: 1
        save_last: false
        monitor: "step"
        mode: "max"
    - class_path: onnx_callback.OnnxExportCallback
      init_args:
        output_dir: "/content/drive/MyDrive/pt/fr/onnx_exports"
        epoch_offset: 0

Основные настройки и главный запуск обучения. Стартовый чекпоинт, если вдруг ещё нет, можно взять здесь https://huggingface.co/rraaww/ru_piper/tree/main

In [ ]:
! PYTHONPATH=/content .venv/bin/python -m piper.train fit \
  --config "/content/callbacks.yaml" \
  --data.voice_name "ru_RU-fr-medium" \
  --data.csv_path "/content/drive/MyDrive/fr/metadata.csv" \
  --data.audio_dir "/content/drive/MyDrive/fr/" \
  --model.sample_rate 22050 \
  --data.espeak_voice "ru" \
  --data.cache_dir "/content/cache/" \
  --data.config_path "/content/drive/MyDrive/ru_RU-fr-medium.json" \
  --data.batch_size 20 \
  --data.validation_split 0.04 \
  --data.num_test_examples 3 \
  --model.learning_rate 0.00007 \
  --model.learning_rate_d 0.00007 \
  --ckpt_path "/content/drive/MyDrive/pt/fr/6400.ckpt" \
  --model.p_dropout 0.12 \
  --trainer.check_val_every_n_epoch 50 \
  --trainer.log_every_n_steps 1 \
  --data.num_workers 2

Настройки p_dropout и двух learning_rate полезны при дообучении. Если при стандартных значениях через 1000-1500 эпох наступает черезмерная металлизация - стоит ознакомиться на что влияют эти параметры, можно ожидать небольшие улучшения. Но качество датасета - первично. Также полезно иметь чекпоинт (промежуточный), на который можно откатывать и пробовать ещё раз с другими значениями.

Весь процесс будет выполняться автоматически, главное не закрывайте браузер, и не выключайте пекарню. При перезапуске обучения не забывайте выполнять менеджмент имён в каталоге, куда сохраняется чекпоинт.

Если требуется отчистить кэш

In [ ]:
!rm -rf /content/cache